# 《LangGraph 使用底层API实现ReACT智能体实验》

## 一、实验目的
1. 理解 Human-In-Loop 的概念与典型场景
2. 理解 Time Travel 的概念与使用场景
3. 掌握使用底层 API 从零构建 ReACT 智能体
4. 完成带人工审核的 AI 助手工作流

## 二、实验环境
- 系统：Windows 10
- Python版本：3.10
- 虚拟环境：Miniconda
- 开发工具：VS Code / Jupyter Notebook
- 依赖库：langgraph、langchain、langchain-core、langchain-deepseek、python-dotenv、loguru
- 模型：DeepSeek（deepseek-chat）

## 三、实验原理

### 3.1 Human-In-Loop 人在环中

在LangGraph 是个“有状态图（state graph）”的工作流框架，每个节点可以是：

- 模型调用（LLM、工具调用）
- 程序逻辑
- 人类反馈（HITL）
HITL 就是把“人类操作”当作图里的一个节点（step），在运行时需要停下来等待人类确认、输入或决策，然后再继续执行。
#### 典型场景
1. 确认操作：比如智能体打算删除数据或执行危险操作 → 先让人确认（Yes/No）。
2. 信息补充：模型缺少必要参数时，提示人类补全，比如填写 API Key、选择文件。
3. 审核 / 修改：模型生成的回答需要人审查、修改，再提交给用户。
4. 主动决策分支：工作流里分叉走向不确定 → 由人来选择接下来走哪条分支。
#### 实现要点
1. 必须指定一个checkpoint短期记忆，否则无法保存任务状态。
2. 在执行Graph任务时，必须指定一个带有thread_id的配置项，指定线程ID。之后才能通过线程ID，指定恢复线程。
3. 在任务执行过程中，通过interrupt()方法，中断任务，等待确认。
4. 在人类确认之后，使用Graph提交一个resume=True的Command指令，恢复任务，并继续进行。
示例代码

### 3.2 Time Travel 时间回溯

在 LangGraph 中，Time Travel 是一个允许你“回到对话的某个历史状态点，并从那里重新执行”的功能。
它依赖 Checkpointer（检查点系统），比如 MemorySaver、数据库持久化 saver 等，把每一步执行的 状态（state） 保存下来。

可以类比成：

- 普通对话：只能按顺序走下去
- 有时间回溯：可以跳到某一步（比如第 3 次工具调用前），从那个状态继续，甚至尝试不同的分支
### 使用场景
* 调试：想看 agent 在某个历史状态下会如何响应
* 修复：发现某一步错误，可以回到那一步，重新走另一条路径
* 探索分支：从同一个历史状态，分叉出多个可能的结果，做 what-if 实验
* 人类在环 (HITL)：如果用户拒绝了工具调用，可以退回到之前状态，重新走对话
### 实现要点
* 在运行 Graph 时，需要提供初始的输入消息。
* 运行时，指定 thread_id 线程 ID。并且要基于这个线程 ID，再指定一个 checkpoint 检查点。执行后将在每一个 Node 执行后，生成一个 check_point_id
* 指定 thread_id 和 check_point_id，进行任务重演。重演前，可以选择更新 state，当然，如果没问题，也可以不指定。

## 四、实验内容

### 4.1 环境准备

In [ ]:
pip install langgraph

### 4.2 Human-In-Loop 示例

In [ ]:
from typing import TypedDict, Annotated, Literal
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.constants import START, END
from langgraph.graph import add_messages, StateGraph
from langgraph.types import Command
import os
import dotenv
from langchain.chat_models import init_chat_model

# 加载环境变量
dotenv.load_dotenv()

# 定义状态结构
class AgentState(TypedDict):
    messages: Annotated[list, add_messages]

# 初始化大模型
llm = init_chat_model(
    "deepseek-chat",
    model_provider="deepseek",
    api_key=os.getenv("DEEPSEEK_API_KEY")
)

# AI 聊天节点
def chatbot(state: AgentState):
    return {"messages": [llm.invoke(state['messages'])]}

# 人工审批节点
def human_approval(state: AgentState) -> Command[Literal["chatbot", END]]:
    question = "是否同意调用大语言模型？(y/n): "
    while True:
        response = input(question).strip().lower()
        if response in ("y", "yes"):
            return Command(goto="chatbot")
        elif response in ("n", "no"):
            print("❌ 已拒绝，流程结束。")
            return Command(goto=END)
        else:
            print("⚠️ 请输入 y 或 n。")

# 构建工作流
graph_builder = StateGraph(AgentState)
graph_builder.add_node("human_approval", human_approval)
graph_builder.add_node("chatbot", chatbot)

graph_builder.add_edge(START, "human_approval")
graph_builder.add_edge("chatbot", END)

# 编译工作流
checkpointer = InMemorySaver()
graph = graph_builder.compile(checkpointer=checkpointer)

# ======================
# 🔥 纯字符流程图（无依赖，不报错）
# ======================
print("\n" + "="*60)
print("📊 工作流字符流程图")
print("="*60)
# 核心：终端打印字符图
print(graph.get_graph().draw_ascii())
print("="*60 + "\n")

# 执行流程
if __name__ == "__main__":
    config = {"configurable": {"thread_id": "chat-1"}}
    result = graph.invoke({"messages": ["北京天气怎么样"]}, config=config)
    
    # 打印最终结果
    if "messages" in result and len(result["messages"]) > 0:
        print("\n✅ 最终回复：", result["messages"][-1].content)

### 4.3 最简单的聊天机器人
我们先定义一个基于本地大语言模型的聊天机器人。chatbot 函数接收当前对话状态，调用 llm.invoke() 生成回复，并返回新消息。整体通过 StateGraph 构建工作流，实现从开始到结束的自动对话处理。

代码如下

In [ ]:
### 最简单的聊天机器人
from typing import TypedDict, Annotated
from langgraph.constants import START, END
from langgraph.graph import add_messages, StateGraph
import os
import dotenv
from langchain.chat_models import init_chat_model
dotenv.load_dotenv(override=True)

# 定义 Agent 的状态结构，包含消息列表
class AgentState(TypedDict):
    messages: Annotated[list, add_messages]


# 初始化本地大语言模型，配置模型名称和推理模式
llm = init_chat_model(
    "deepseek-chat",
    model_provider="deepseek",
    api_key=os.getenv("DEEPSEEK_API_KEY")
)


# 聊天机器人函数，用于处理对话状态并生成回复
def chatbot(state: AgentState):
    return {"messages": [llm.invoke(state['messages'])]}


# 构建状态图结构
graph_builder = StateGraph(AgentState)

# 每个节点都与对应的处理函数进行绑定，构成工作流的基本单元
graph_builder.add_node("chatbot", chatbot)

# 添加边：从 START 到 chatbot，然后到 END
graph_builder.add_edge(START, "chatbot")
graph_builder.add_edge("chatbot",END)


# 编译图结构，并绘制可视化图表
graph = graph_builder.compile()
print("\n" + "="*50)
print("工作流字符流程图")
print("="*50)
print(graph.get_graph().draw_ascii())
print("="*50 + "\n")

response1 = graph.invoke({"messages": ["北京天气怎么样"]})

print(response1["messages"][-1].content)

### 4.4 添加提示词和工具

In [ ]:
import json
import os
import dotenv
from loguru import logger
from pydantic import Field, BaseModel
from langchain_core.tools import tool

# 加载环境变量配置
dotenv.load_dotenv()


class WeatherQuery(BaseModel):
    """
    天气查询参数模型类，用于定义天气查询工具的输入参数结构。

    :param city: 城市名称，字符串类型，表示要查询天气的城市
    """
    city: str = Field(description="城市名称")


class WriteQuery(BaseModel):
    """
    写入查询模型类

    用于定义需要写入文档的内容结构，继承自BaseModel基类

    属性:
        content (str): 需要写入文档的具体内容，包含详细的描述信息
    """
    content: str = Field(description="需要写入文档的具体内容")


@tool(args_schema=WeatherQuery)
def get_weather(city):
    """
    查询指定城市的即时天气信息（模拟数据，用于学习演示）。

    :param city: 必要参数，字符串类型，表示要查询天气的城市名称。
    :return: 返回模拟的天气数据 JSON 格式字符串。
    """
    # 模拟天气数据
    mock_data = {
        "Beijing": {"city": "北京", "temp": 25, "weather": "晴", "humidity": 45},
        "Shanghai": {"city": "上海", "temp": 22, "weather": "多云", "humidity": 65},
        "Guangzhou": {"city": "广州", "temp": 28, "weather": "雷阵雨", "humidity": 80},
        "Shenzhen": {"city": "深圳", "temp": 27, "weather": "多云", "humidity": 75},
        "Hangzhou": {"city": "杭州", "temp": 23, "weather": "阴", "humidity": 60},
    }
    
    # 尝试匹配城市名（支持中英文）
    for key, data in mock_data.items():
        if key.lower() == city.lower() or data["city"] in city:
            result = json.dumps(data, ensure_ascii=False)
            logger.info(f"查询天气结果：{result}")
            return result
    
    # 默认返回北京天气
    default_data = {"city": city, "temp": 25, "weather": "晴", "humidity": 50}
    result = json.dumps(default_data, ensure_ascii=False)
    logger.info(f"查询天气结果（默认）：{result}")
    return result


@tool(args_schema=WriteQuery)
def write_file(content):
    """
    将指定内容写入本地文件

    参数:
        content (str): 要写入文件的文本内容

    返回值:
        str: 表示写入操作成功完成的提示信息
    """
    # 将内容写入res.txt文件，使用utf-8编码确保中文字符正确保存
    with open('res.txt', 'w', encoding='utf-8') as f:
        f.write(content)
        logger.info(f"已成功写入本地文件，写入内容：{content}")
        return "已成功写入本地文件。"

### 4.5 构建ReACT Agent

In [ ]:
from typing import TypedDict, Annotated
from langchain_core.messages import SystemMessage
from langgraph.constants import START, END
from langgraph.graph import add_messages, StateGraph
from langgraph.prebuilt import ToolNode
import os
import dotenv
from langchain.chat_models import init_chat_model

# 加载环境变量
dotenv.load_dotenv(override=True)

# 定义 Agent 的状态结构，包含消息列表
class AgentState(TypedDict):
    messages: Annotated[list, add_messages]


# 初始化大语言模型
llm = init_chat_model(
    "deepseek-chat",
    model_provider="deepseek",
    api_key=os.getenv("DEEPSEEK_API_KEY")
)
tools = [get_weather, write_file]
llm_with_tools = llm.bind_tools(tools)


# 聊天机器人节点，用于处理对话状态并生成回复，并告诉模型可以调用哪些工具
def chat_node(state: AgentState):
    messages = state["messages"]
    system_prompt = """你是一个智能助手，具备以下能力：
                    1. 查询天气信息
                    2. 结果写入文件
                    请根据用户的需求，选择合适的工具来完成任务。回答要准确、友好、专业。"""
    # 构建完整的消息列表（系统提示词 + 用户消息）,如果第一条消息不是系统消息，则添加系统提示词
    if not any(isinstance(msg, SystemMessage) for msg in messages):
        messages = [SystemMessage(
            content=system_prompt)] + messages
    result = llm_with_tools.invoke(messages)
    return {"messages": [result]}


# 定义工具节点（系统预置 ToolNode 会自动解析 tool_calls）
tool_node = ToolNode(tools=tools)


# 动态路由：chat_node → tool_node 或 END
def route_after_chat(state: AgentState):
    """判断是否需要进入工具节点"""
    last_message = state["messages"][-1]
    if hasattr(last_message, "tool_calls") and last_message.tool_calls:
        return "tool_node"
    return END


# 构建状态图结构
graph_builder = StateGraph(AgentState)

# 每个节点都与对应的处理函数进行绑定，构成工作流的基本单元
graph_builder.add_node("chat_node", chat_node)
graph_builder.add_node("tool_node", tool_node)

# 添加边：从 START 到 chatbot，然后到 END
graph_builder.add_edge(START, "chat_node")
# 添加条件边：根据是否有工具调用来判断是否需要进入工具节点
graph_builder.add_conditional_edges("chat_node", route_after_chat, ["tool_node", END])
# 工具节点执行完后回到 chat_node，继续多轮对话
graph_builder.add_edge("tool_node", "chat_node")

# 编译图结构，并绘制可视化图表
graph = graph_builder.compile()
print("\n" + "="*50)
print("工作流字符流程图")
print("="*50)
print(graph.get_graph().draw_ascii())
print("="*50 + "\n")

response1 = graph.invoke({"messages": ["北京天气怎么样"]})

print(response1["messages"][-1].content)

### 4.6 添加HITL环节

In [ ]:
from typing import TypedDict, Annotated
import json
from langchain_core.messages import SystemMessage, ToolMessage
from langgraph.constants import START, END
from langgraph.graph import add_messages, StateGraph
from langgraph.prebuilt import ToolNode
from langgraph.checkpoint.memory import MemorySaver
import os
import dotenv
from langchain.chat_models import init_chat_model

# 加载环境变量
dotenv.load_dotenv(override=True)


# 定义 Agent 的状态结构
class AgentState(TypedDict):
    messages: Annotated[list, add_messages]


# 初始化本地大语言模型
llm = init_chat_model(
    "deepseek-chat",
    model_provider="deepseek",
    api_key=os.getenv("DEEPSEEK_API_KEY")
)
tools = [get_weather, write_file]
llm_with_tools = llm.bind_tools(tools)


# 聊天机器人节点
def chat_node(state: AgentState):
    messages = state["messages"]
    system_prompt = """你是一个智能助手，具备以下能力：
                    1. 查询天气信息
                    2. 结果写入文件
                    请根据用户的需求，选择合适的工具来完成任务。回答要准确、友好、专业。"""

    if not any(isinstance(msg, SystemMessage) for msg in messages):
        messages = [SystemMessage(content=system_prompt)] + messages

    result = llm_with_tools.invoke(messages)
    return {"messages": [result]}


# 定义工具节点
tool_node = ToolNode(tools=tools)


# 动态路由：chat_node 之后
def route_after_chat(state: AgentState):
    """判断是否需要调用工具"""
    last_message = state["messages"][-1]
    if hasattr(last_message, "tool_calls") and last_message.tool_calls:
        return "tool_node"
    return END


# 构建状态图
graph_builder = StateGraph(AgentState)

# 添加节点
graph_builder.add_node("chat_node", chat_node)
graph_builder.add_node("tool_node", tool_node)

# 添加边
graph_builder.add_edge(START, "chat_node")
graph_builder.add_conditional_edges("chat_node", route_after_chat, ["tool_node", END])
graph_builder.add_edge("tool_node", "chat_node")

# 编译图结构 - 关键：使用 interrupt_before 在工具节点前中断
memory = MemorySaver()
graph = graph_builder.compile(
    checkpointer=memory,
    interrupt_before=["tool_node"]  # 在执行工具前中断，等待人工确认
)


def run_with_approval():
    """运行带人工确认的工作流"""
    config = {"configurable": {"thread_id": "1"}}

    # 第一步：发送用户消息，执行到中断点
    print("【用户】北京天气怎么样\n")
    result = graph.invoke({"messages": ["北京天气怎么样"]}, config)

    # 检查是否中断（等待人工确认）
    snapshot = graph.get_state(config)

    if snapshot.next:  # 如果有下一个节点，说明被中断了
        print("工具调用需要人工确认")

        # 获取待执行的工具调用信息
        last_message = snapshot.values["messages"][-1]
        if hasattr(last_message, "tool_calls") and last_message.tool_calls:
            for idx, tool_call in enumerate(last_message.tool_calls, 1):
                print(f"\n[{idx}] 工具名称: {tool_call['name']}")
                print(f"    调用参数: {json.dumps(tool_call['args'], ensure_ascii=False, indent=4)}")

            approval = input("是否批准执行？(yes/no): ").strip().lower()

            if approval in ['yes', 'y']:
                print("工具调用已批准，继续执行...\n")
                # 继续执行（resume）
                result = graph.invoke(None, config)

            elif approval in ['no', 'n']:
                print("工具调用已被拒绝\n")
                # 手动添加拒绝消息，然后继续
                tool_messages = []
                for tool_call in last_message.tool_calls:
                    tool_messages.append(
                        ToolMessage(
                            content="工具调用被用户拒绝，请询问用户是否需要调整方案或提供更多信息。",
                            tool_call_id=tool_call["id"]
                        )
                    )
                # 更新状态并跳过工具节点
                graph.update_state(config, {"messages": tool_messages})
                result = graph.invoke(None, config)

    # 输出最终结果
    print("最终回复:")
    final_message = result["messages"][-1]
    print(final_message.content if hasattr(final_message, 'content') else str(final_message))


# 测试运行
if __name__ == "__main__":
    run_with_approval()

## 五、实验总结

1. 理解了 Human-In-Loop 的概念与实现方法
2. 理解了 Time Travel 的概念与 Checkpointer 的作用
3. 从零构建了 ReACT 智能体
4. 通过 interrupt_before 实现了工具调用前的人工审核
5. 掌握了 MemorySaver 在状态持久化中的作用